# torch_concepts Mid-Level API — Deep Dive

This notebook is a comprehensive walkthrough of `torch_concepts.nn.modules.mid`, the probabilistic graphical modelling layer that sits between low-level PyTorch/Pyro primitives and the high-level Lightning-based training API.

**Purpose:** Help a contributor deeply understand every component so the API can be refactored confidently.

---

## Outline

1. [Architecture Overview](#1-architecture-overview)
2. [Variables](#2-variables) — typed random variable nodes
3. [ParametricCPD](#3-parametriccpd) — neural-net-backed conditional probability distributions
4. [BayesianNetwork / ProbabilisticModel](#4-bayesiannetwork--probabilisticmodel) — the DAG container
5. [ForwardInference (abstract base)](#5-forwardinference-abstract-base) — topological forward loop
6. [DeterministicInference](#6-deterministicinference) — training via point estimates
7. [AncestralSamplingInference](#7-ancestralsamplinginference) — stochastic generative queries
8. [IndependentInference](#8-independentinference) — independent / CEM-style training
9. [ELBOInference & AmortizedGuide](#9-elboinference--amortizedguide) — variational training
10. [Posterior Query Engines](#10-posterior-query-engines) — ImportanceQuery, ExactDiscreteQuery, MCMCQuery
11. [InferenceOutput](#11-inferenceoutput) — the structured return type
12. [End-to-End Example: Asia Bayesian Network](#12-end-to-end-example-asia-bayesian-network)
13. [Key Design Tensions & Refactoring Notes](#13-key-design-tensions--refactoring-notes)

In [1]:
import sys
sys.path.insert(0, '/home/francesco/projects/pytorch_concepts')

import torch
import torch.nn as nn
from torch.distributions import Bernoulli, Normal, OneHotCategorical, RelaxedBernoulli, MultivariateNormal
import pyro
import pyro.distributions as pydist

# just fo displaying the notebook in a better way
from IPython.display import HTML, display

pyro.set_rng_seed(42)
torch.manual_seed(42)
pyro.settings.set(module_local_params=True)

print('torch  :', torch.__version__)
print('pyro   :', pyro.__version__)

torch  : 2.5.1
pyro   : 1.9.1


---
## 1. Architecture Overview

The mid-level API implements concept-based probabilistic graphical models using four orthogonal layers:


<style>
.arch-wrap { font-family: 'SF Mono', 'Fira Code', 'Consolas', monospace; max-width: 700px; padding: 8px 0; }
.arch-layer { border-radius: 10px; margin-bottom: 10px; overflow: hidden; border: 1px solid; }
.layer-header { display: flex; align-items: baseline; gap: 10px; padding: 10px 14px 6px; }
.layer-num { font-size: 10px; font-weight: 700; letter-spacing: .08em; padding: 2px 7px; border-radius: 4px; white-space: nowrap; }
.layer-title { font-size: 14px; font-weight: 700; }
.layer-path { font-size: 10px; opacity: .55; margin-left: auto; white-space: nowrap; overflow: hidden; text-overflow: ellipsis; max-width: 320px; }
.layer-body { padding: 0 14px 10px; }
.class-pills { display: flex; flex-wrap: wrap; gap: 5px; margin-bottom: 6px; }
.pill { font-size: 11.5px; padding: 2px 9px; border-radius: 20px; white-space: nowrap; border: 1px solid; }
.layer-desc { font-size: 11px; opacity: .6; font-style: italic; margin-top: 4px; }
.l1 { background: #f0eeff; border-color: #c5baff; }
.l1 .layer-header { background: #e6e1ff; }
.l1 .layer-num { background: #7c6bff; color: #fff; }
.l1 .layer-title { color: #3d2fa0; }
.l1 .pill { background: #fff; border-color: #b0a0ff; color: #3d2fa0; }
.l1 .layer-desc { color: #3d2fa0; }
.l2 { background: #edfaf5; border-color: #7fd9bc; }
.l2 .layer-header { background: #d5f5ea; }
.l2 .layer-num { background: #1d9e75; color: #fff; }
.l2 .layer-title { color: #0a5c40; }
.l2 .pill { background: #fff; border-color: #7fd9bc; color: #0a5c40; }
.l2 .layer-desc { color: #0a5c40; }
.l3 { background: #fff9ee; border-color: #fac775; }
.l3 .layer-header { background: #fff0d0; }
.l3 .layer-num { background: #d08000; color: #fff; }
.l3 .layer-title { color: #7a4900; }
.l3 .pill { background: #fff; border-color: #fac775; color: #7a4900; }
.l3 .layer-desc { color: #7a4900; }
.l4 { background: #fff1ee; border-color: #f0997b; }
.l4 .layer-header { background: #ffe4de; }
.l4 .layer-num { background: #c84420; color: #fff; }
.l4 .layer-title { color: #7a2010; }
.l4 .pill { background: #fff; border-color: #f0997b; color: #7a2010; }
.l4 .pill.accent { background: #c84420; color: #fff; border-color: #c84420; }
.l4 .layer-desc { color: #7a2010; }
.inf-tree { display: flex; flex-direction: column; gap: 4px; margin-top: 2px; }
.inf-group { display: flex; align-items: flex-start; gap: 6px; }
.inf-branch { display: flex; flex-direction: column; padding-left: 12px; border-left: 2px solid #f0997b; margin-left: 6px; gap: 3px; margin-top: 3px; }
.inf-sub { display: flex; align-items: center; gap: 5px; }
.inf-sub-pill { font-size: 11px; padding: 2px 8px; border-radius: 20px; background: #fff; border: 1px solid #f0997b; color: #7a2010; white-space: nowrap; }
.inf-sub-desc { font-size: 10.5px; opacity: .55; color: #7a2010; }
.inf-standalone { display: flex; flex-wrap: wrap; gap: 5px; margin-top: 6px; }
.inf-sep { font-size: 11px; opacity: .35; color: #7a2010; margin-top: 5px; }
</style>
<div class="arch-wrap">
  <div class="arch-layer l1">
    <div class="layer-header"><span class="layer-num">LAYER 1</span><span class="layer-title">Variables</span><span class="layer-path">torch_concepts/nn/modules/mid/models/variable.py</span></div>
    <div class="layer-body">
      <div class="class-pills"><span class="pill">Variable</span><span class="pill">ConceptVariable</span><span class="pill">LatentVariable</span><span class="pill">ExogenousVariable</span><span class="pill">EndogenousVariable</span></div>
      <div class="layer-desc">→ typed random-variable nodes carrying distribution metadata</div>
    </div>
  </div>
  <div class="arch-layer l2">
    <div class="layer-header"><span class="layer-num">LAYER 2</span><span class="layer-title">Factors</span><span class="layer-path">torch_concepts/nn/modules/mid/models/cpd.py</span></div>
    <div class="layer-body">
      <div class="class-pills"><span class="pill">ParametricCPD</span></div>
      <div class="layer-desc">→ PyroModule wrapping a neural-net parametrization + parent list</div>
    </div>
  </div>
  <div class="arch-layer l3">
    <div class="layer-header"><span class="layer-num">LAYER 3</span><span class="layer-title">Model</span><span class="layer-path">torch_concepts/nn/modules/mid/models/probabilistic_model.py</span></div>
    <div class="layer-body">
      <div class="class-pills"><span class="pill">BayesianNetwork</span><span class="pill">ProbabilisticModel</span><span class="pill">_ProbabilisticModelBase</span></div>
      <div class="layer-desc">→ PyroModule DAG container; topological sort; Pyro generative forward()</div>
    </div>
  </div>
  <div class="arch-layer l4">
    <div class="layer-header"><span class="layer-num">LAYER 4</span><span class="layer-title">Inference</span><span class="layer-path">torch_concepts/nn/modules/mid/inference/</span></div>
    <div class="layer-body">
      <div class="inf-tree">
        <div class="inf-group"><span class="pill accent">ForwardInference</span><span class="layer-desc" style="margin-top:4px;font-size:10.5px;">abstract base</span></div>
        <div class="inf-branch">
          <div class="inf-sub"><span class="inf-sub-pill">DeterministicInference</span><span class="inf-sub-desc">point-estimate forward pass</span></div>
          <div class="inf-sub"><span class="inf-sub-pill">AncestralSamplingInference</span><span class="inf-sub-desc">stochastic forward sampling</span></div>
          <div class="inf-sub"><span class="inf-sub-pill">IndependentInference</span><span class="inf-sub-desc">p=1 GT-propagation variant</span></div>
        </div>
        <div class="inf-sep">── standalone engines ──</div>
        <div class="inf-standalone">
          <div class="inf-sub"><span class="pill">ELBOInference</span><span class="inf-sub-desc" style="margin-left:4px;">Pyro ELBO variational training</span></div>
          <div class="inf-sub"><span class="pill">AmortizedGuide</span><span class="inf-sub-desc" style="margin-left:4px;">auto-built amortized guide</span></div>
          <div class="inf-sub"><span class="pill">ImportanceQuery</span><span class="inf-sub-desc" style="margin-left:4px;">importance-resampling posterior</span></div>
          <div class="inf-sub"><span class="pill">ExactDiscreteQuery</span><span class="inf-sub-desc" style="margin-left:4px;">variable-elimination posterior</span></div>
          <div class="inf-sub"><span class="pill">MCMCQuery</span><span class="inf-sub-desc" style="margin-left:4px;">NUTS/HMC posterior</span></div>
        </div>
      </div>
    </div>
  </div>
</div>


All inference is **external to the model** — the `BayesianNetwork` holds no inference state. Swap engines without rebuilding the model.

The `BayesianNetwork.forward()` is a full Pyro generative program: every variable becomes a `pyro.sample` (or `pyro.deterministic`) site, enabling Pyro's effect-handler ecosystem (`poutine.condition`, `Predictive`, ELBO, MCMC, etc.) to work out of the box.

---
## 2. Variables

**File:** `torch_concepts/nn/modules/mid/models/variable.py`

A `Variable` is a node in the PGM. It carries:
- `concept` (str) — unique name used as the Pyro sample-site name
- `distribution` — a PyTorch/Pyro distribution class
- `size` — event dimension (e.g., 1 for binary, k for categorical)
- `dist_kwargs` — extra constructor kwargs (e.g., `{'temperature': 0.5}` for relaxed distributions)
- `_observed` — optional override of the `is_observed` property

The constructor accepts either `concept=<str>` (single variable) or `concepts=[<str>, ...]` (batch construction → returns a `list` of variables, one per name). The two are mutually exclusive.

> **Note.** Activation functions have been removed from variables. To map raw distribution parameters to probabilities, use `variable.make_distribution(params).mean` (or `.probs` / sampling APIs) — the variable owns the parameter→distribution mapping, and the distribution owns the conversion to probabilities.

### 2.1 Variable subclasses

| Class | Default `is_observed` | Typical use |
|---|---|---|
| `Variable` | depends on metadata | base class; rarely used directly |
| `ConceptVariable` | `False` | supervisable concept nodes (optional GT) |
| `LatentVariable` | `False` | hidden/unsupervised latents |
| `ExogenousVariable` | `True` | root input nodes (always in evidence) |
| `EndogenousVariable` | alias for `ConceptVariable` | backward compat |
| `InputVariable` | alias for `LatentVariable` | backward compat |


In [ ]:
from torch_concepts.nn.modules.mid.models.variable import (
    Variable, ConceptVariable, ExogenousVariable, LatentVariable,
    param_dim, _SUPPORTED_DISTRIBUTIONS, _PARAM_DIMS,
)
from torch_concepts.distributions import Delta

# Single variable construction — use `concept=` (str)
input_var = LatentVariable(concept='input', distribution=Delta, size=32)
asia_var  = ConceptVariable(concept='asia',  distribution=Bernoulli, size=1)
color_var = ConceptVariable(concept='color', distribution=OneHotCategorical, size=3)  # 3-class
cont_var  = ConceptVariable(concept='cont',  distribution=Normal, size=4)

print('input_var :', input_var)
print('asia_var  :', asia_var)
print('color_var :', color_var)
print('cont_var  :', cont_var)


/home/francesco/projects/pytorch_concepts/torch_concepts/nn/modules/high/base/model.py:39: FutureWarning: The 'torch_concepts.nn.mid' module contains experimental APIs that are unstable and subject to change without notice. If you are using these classes intentionally, be aware that breaking changes may occur in future releases. Consider using the high-level API (torch_concepts.nn.high) for stable interfaces.
  from ...mid.constructors.concept_graph import ConceptGraph


input_var : Variable(concept='input', dist=Delta, size=32, param_dim={'value': 32}, metadata={'variable_type': 'latent'})
asia_var  : Variable(concept='asia', dist=Bernoulli, size=1, param_dim={'logits': 1}, metadata={'variable_type': 'concept'})
color_var : Variable(concept='color', dist=OneHotCategorical, size=3, param_dim={'logits': 3}, metadata={'variable_type': 'concept'})
cont_var  : Variable(concept='cont', dist=Normal, size=4, param_dim={'loc': 4, 'scale': 4}, metadata={'variable_type': 'concept'})


In [3]:
print("Batch construction of discrete variables: passing `concepts=[...]` → returns a LIST of Variable objects")
batch_vars = ConceptVariable(concepts=['c1', 'c2', 'c3'], distribution=Bernoulli, size=1)
print(type(batch_vars), len(batch_vars))
for v in batch_vars:
    print(' ', v.__dict__)

print("Batch construction of continuous variables: passing `concepts=[...]` → returns a LIST of Variable objects")
batch_vars = ConceptVariable(concepts=['c1', 'c2', 'c3'], distribution=Normal, size=1)
print(type(batch_vars), len(batch_vars))
for v in batch_vars:
    print(' ', v.__dict__)


Batch construction of discrete variables: passing `concepts=[...]` → returns a LIST of Variable objects
<class 'list'> 3
  {'concept': 'c1', 'distribution': <class 'torch.distributions.bernoulli.Bernoulli'>, 'size': 1, 'dist_kwargs': {}, 'metadata': {'variable_type': 'concept'}, '_observed': None}
  {'concept': 'c2', 'distribution': <class 'torch.distributions.bernoulli.Bernoulli'>, 'size': 1, 'dist_kwargs': {}, 'metadata': {'variable_type': 'concept'}, '_observed': None}
  {'concept': 'c3', 'distribution': <class 'torch.distributions.bernoulli.Bernoulli'>, 'size': 1, 'dist_kwargs': {}, 'metadata': {'variable_type': 'concept'}, '_observed': None}
Batch construction of continuous variables: passing `concepts=[...]` → returns a LIST of Variable objects
<class 'list'> 3
  {'concept': 'c1', 'distribution': <class 'torch.distributions.normal.Normal'>, 'size': 1, 'dist_kwargs': {}, 'metadata': {'variable_type': 'concept'}, '_observed': None}
  {'concept': 'c2', 'distribution': <class 'torch.

### 2.2 Supported distributions and parameter dimensions

The `param_dim()` helper tells how many raw scalars a CPD must output for each distribution/size combination. This is used internally to size linear layers.

In [4]:
import pandas as pd

rows = []
for dist in _SUPPORTED_DISTRIBUTIONS:
    size = 1 if dist not in [OneHotCategorical, Normal, MultivariateNormal] else 4
    total = param_dim(dist, size=size, return_sum=True)
    breakdown = param_dim(dist, size=size, return_sum=False)
    rows.append({'Distribution': dist.__name__, 'size': size,
                 'total param_dim': total, 'breakdown': str(breakdown)})

pd.DataFrame(rows).set_index('Distribution')

,size,total param_dim,breakdown
Distribution,,,
Bernoulli,1,1,{'logits': 1}
RelaxedBernoulli,1,1,{'logits': 1}
OneHotCategorical,4,4,{'logits': 4}
RelaxedOneHotCategorical,1,1,{'logits': 1}
Normal,4,8,"{'loc': 4, 'scale': 4}"
MultivariateNormal,4,14,"{'loc': 4, 'scale_tril': 10}"
Delta,1,1,{'value': 1}


### 2.3 Key Variable properties

In [5]:
print('--- asia_var (ConceptVariable, Bernoulli) ---')
print('  concept         :', asia_var.concept)
print('  pyro_site_name  :', asia_var.pyro_site_name)  # == concept
print('  is_observed     :', asia_var.is_observed)      # False — optional GT
print('  is_deterministic:', asia_var.is_deterministic) # False — has a sampled dist
print('  out_features    :', asia_var.out_features)     # raw param_dim total
print('  param_dim dict  :', asia_var.param_dim)        # per-param breakdown

print()
print('--- input_var (LatentVariable, Delta) ---')
print('  is_deterministic:', input_var.is_deterministic) # True → pyro.deterministic()


--- asia_var (ConceptVariable, Bernoulli) ---
  concept         : asia
  pyro_site_name  : asia
  is_observed     : False
  is_deterministic: False
  out_features    : 1
  param_dim dict  : {'logits': 1}

--- input_var (LatentVariable, Delta) ---
  is_deterministic: True


### 2.4 `make_distribution()` — building Pyro distributions from raw network params

`variable.make_distribution(params)` is the bridge between a CPD's raw neural-network output and a Pyro sample site. It accepts either a single `torch.Tensor` (concatenated raw parameters) **or** a `Dict[str, Tensor]` keyed by parameter name (e.g. `{'loc': ..., 'scale': ...}`). The rules:

<style>
.dist-wrap { font-family: 'SF Mono', 'Fira Code', 'Consolas', monospace; max-width: 720px; display: flex; flex-direction: column; gap: 7px; padding: 8px 0; }
.dist-row { display: grid; grid-template-columns: 150px 1fr 1fr; align-items: center; border-radius: 8px; overflow: hidden; border: 1px solid; }
.dist-row:hover { filter: brightness(0.97); }
.dist-name { padding: 9px 12px; font-size: 12.5px; font-weight: 700; border-right: 1px solid; }
.dist-shape { padding: 9px 12px; font-size: 11.5px; border-right: 1px solid; display: flex; flex-direction: column; gap: 2px; }
.dist-pyro { padding: 9px 12px; font-size: 11px; }
.shape-badge { display: inline-block; padding: 1px 7px; border-radius: 4px; font-size: 10.5px; font-weight: 700; letter-spacing: .03em; margin-bottom: 2px; }
.shape-note { font-size: 10px; opacity: .55; font-style: italic; }
.col-header { display: grid; grid-template-columns: 150px 1fr 1fr; padding: 0 0 4px; }
.col-header span { font-size: 10px; letter-spacing: .07em; text-transform: uppercase; opacity: .45; padding: 0 12px; }
em.kw { font-style: normal; opacity: .5; }
.r-delta  { background:#f6f4ff;border-color:#c9c0ff; } .r-delta  .dist-name{background:#ede9ff;border-color:#c9c0ff;color:#3a28aa;} .r-delta  .dist-shape,.r-delta  .dist-pyro{border-color:#ddd7ff;color:#3a28aa;} .r-delta  .shape-badge{background:#7c6bff;color:#fff;}
.r-bern   { background:#edfaf6;border-color:#7fd9bc; } .r-bern   .dist-name{background:#d5f5ea;border-color:#7fd9bc;color:#0a5c40;} .r-bern   .dist-shape,.r-bern   .dist-pyro{border-color:#b8edda;color:#0a5c40;} .r-bern   .shape-badge{background:#1d9e75;color:#fff;}
.r-rbern  { background:#f0fdf8;border-color:#a3e6cc; } .r-rbern  .dist-name{background:#d5f5ea;border-color:#a3e6cc;color:#0a5c40;} .r-rbern  .dist-shape,.r-rbern  .dist-pyro{border-color:#c5f0e0;color:#0a5c40;} .r-rbern  .shape-badge{background:#0f9070;color:#fff;}
.r-ohcat  { background:#fff9ee;border-color:#fac775; } .r-ohcat  .dist-name{background:#fff0d0;border-color:#fac775;color:#7a4900;} .r-ohcat  .dist-shape,.r-ohcat  .dist-pyro{border-color:#ffdea0;color:#7a4900;} .r-ohcat  .shape-badge{background:#d08000;color:#fff;}
.r-normal { background:#fff3ee;border-color:#f5b89b; } .r-normal .dist-name{background:#ffe6da;border-color:#f5b89b;color:#7a2800;} .r-normal .dist-shape,.r-normal .dist-pyro{border-color:#ffd0be;color:#7a2800;} .r-normal .shape-badge{background:#d85a30;color:#fff;}
.r-mvn    { background:#fdf0f5;border-color:#f0a0c0; } .r-mvn    .dist-name{background:#fce0eb;border-color:#f0a0c0;color:#7a1040;} .r-mvn    .dist-shape,.r-mvn    .dist-pyro{border-color:#f8c8d8;color:#7a1040;} .r-mvn    .shape-badge{background:#c84070;color:#fff;}
</style>
<div class="dist-wrap">
  <div class="col-header"><span>Distribution</span><span>Input shape (params)</span><span>Pyro distribution built</span></div>
  <div class="dist-row r-delta">
    <div class="dist-name">Delta</div>
    <div class="dist-shape"><span class="shape-badge">{'value': (B, size)}</span></div>
    <div class="dist-pyro">Delta(value, event_dim=1)</div>
  </div>
  <div class="dist-row r-bern">
    <div class="dist-name">Bernoulli</div>
    <div class="dist-shape"><span class="shape-badge">{'logits': (B, 1)}</span></div>
    <div class="dist-pyro">Bernoulli(logits=...)<em class="kw">.to_event(1)</em></div>
  </div>
  <div class="dist-row r-rbern">
    <div class="dist-name">RelaxedBernoulli</div>
    <div class="dist-shape"><span class="shape-badge">{'logits': (B, 1)}</span></div>
    <div class="dist-pyro">RelaxedBernoulli(temperature, logits)<em class="kw">.to_event(1)</em></div>
  </div>
  <div class="dist-row r-ohcat">
    <div class="dist-name">OneHotCategorical</div>
    <div class="dist-shape"><span class="shape-badge">{'logits': (B, k)}</span></div>
    <div class="dist-pyro">OneHotCategorical(logits=...)</div>
  </div>
  <div class="dist-row r-normal">
    <div class="dist-name">Normal</div>
    <div class="dist-shape"><span class="shape-badge">{'loc': (B, size), 'scale': (B, size)}</span></div>
    <div class="dist-pyro">Normal(loc, softplus(scale))<em class="kw">.to_event(1)</em></div>
  </div>
  <div class="dist-row r-mvn">
    <div class="dist-name">MultivariateNormal</div>
    <div class="dist-shape"><span class="shape-badge">{'loc': (B, d), 'scale_tril': (B, d(d+1)/2)}</span></div>
    <div class="dist-pyro">MultivariateNormal(loc, scale_tril=L)</div>
  </div>
</div>

A bare `Tensor` is also accepted: it is split along the last dim into the parameter slots above.


In [6]:
# Demonstrate make_distribution for Bernoulli and Normal
B = 4  # batch size

bernoulli_params = torch.randn(B, 1)           # raw logit
d_bern = asia_var.make_distribution(bernoulli_params)
print('Bernoulli variable')
print(type(d_bern))
print(d_bern)
print('  sample shape  :', d_bern.sample().shape)  # (B, 1)
print()

normal_params = torch.randn(B, 8)              # loc (4) + log-scale (4)
d_norm = cont_var.make_distribution(normal_params)
print('Normal variable')
print(type(d_norm))
print(d_norm)
print('  sample shape  :', d_norm.sample().shape)  # (B, 4)

Bernoulli variable
<class 'pyro.distributions.torch.Independent'>
Independent(Bernoulli(logits: torch.Size([4, 1])), 1)
  sample shape  : torch.Size([4, 1])

Normal variable
<class 'pyro.distributions.torch.Independent'>
Independent(Normal(loc: torch.Size([4, 4]), scale: torch.Size([4, 4])), 1)
  sample shape  : torch.Size([4, 4])


---
## 3. ParametricCPD

**File:** `torch_concepts/nn/modules/mid/models/cpd.py`

`ParametricCPD` is a `PyroModule` that wraps:
- `parametrization` — an `nn.Module` (or `nn.ModuleDict`) computing raw distribution parameters from parent values
- `parents` — ordered list of `Variable` objects (or string names resolved later)
- `concept` (str) **or** `concepts` (List[str]) — the name(s) of the child variable(s) this CPD models. Mutually exclusive: pass `concept=` for a single CPD, `concepts=[...]` for a batch
- `shared` — when `True` together with `concepts=[...]`, builds **one** CPD producing concatenated outputs for all concepts (shared weights). When `False`, `concepts=[...]` returns a *list* of independent CPDs

### 3.1 Construction patterns


In [7]:
from torch_concepts.nn.modules.mid.models.cpd import ParametricCPD

# Pattern 1: Root node — Identity pass-through (no parents)
cpd_input = ParametricCPD(concept='input', parametrization=nn.Identity())
print('root CPD :', cpd_input)

# Pattern 2: Child node — single linear layer
cpd_asia = ParametricCPD(concept='asia', parametrization=nn.Linear(32, 1), parents=[input_var])
print('child CPD:', cpd_asia)


root CPD : ParametricCPD(concept='input', parametrization=Identity, parents=[])
child CPD: ParametricCPD(concept='asia', parametrization=Linear, parents=['input'])


In [8]:
# Pattern 3: batch construction — passing `concepts=[...]` (without shared=True)
# Returns a LIST of deep-copied CPDs (one per concept name)
batch_cpds = ParametricCPD(concepts=['c1', 'c2', 'c3'], parametrization=nn.Linear(32, 1), parents=[input_var])
print(type(batch_cpds), len(batch_cpds))
for cpd in batch_cpds:
    print(' ', cpd.concept, '| parametrization id:', id(cpd.parametrization))  # different ids → deep copies


<class 'list'> 3
  c1 | parametrization id: 138821760230720
  c2 | parametrization id: 138821760229232
  c3 | parametrization id: 138821760225872


In [9]:
# Pattern 4: shared=True — single CPD for multiple concepts, shared weights
# Output dimension must be sum of all concept sizes
shared_cpd = ParametricCPD(
    concepts=['c1', 'c2'],
    parametrization=nn.Linear(32, 2),  # 1 + 1 = 2
    parents=[input_var],
    shared=True,
)
print('shared CPD:', shared_cpd)
print('  .shared    :', shared_cpd.shared)
print('  .concept   :', shared_cpd.concept)    # the canonical name (first concept)
print('  .concepts  :', shared_cpd.concepts)   # full list when shared


shared CPD: ParametricCPD(concepts=['c1', 'c2'], parametrization=Linear, parents=['input'], shared=True)
  .shared    : True
  .concept   : c1
  .concepts  : ['c1', 'c2']


In [10]:
# Pattern 5: dict parametrization — separate networks per distribution parameter
# Useful for Normal: one net for loc, another for scale
normal_cpd = ParametricCPD(
    concept='z',
    parametrization={'loc': nn.Linear(32, 4), 'scale': nn.Linear(32, 4)},
    parents=[input_var],
)
print('dict CPD:', normal_cpd)
print('  .parametrization:', type(normal_cpd.parametrization))

# forward() returns a Dict[str, Tensor] keyed by parameter name when the
# parametrization is a ModuleDict (multi-parameter distributions).
x = torch.randn(4, 32)
out = normal_cpd(x)
print('  output type :', type(out).__name__)
print('  output keys :', list(out.keys()))
print('  loc shape   :', out['loc'].shape)    # (4, 4)
print('  scale shape :', out['scale'].shape)  # (4, 4)


dict CPD: ParametricCPD(concept='z', parametrization={loc: Linear, scale: Linear}, parents=['input'])
  .parametrization: <class 'torch.nn.modules.container.ModuleDict'>
  output type : dict
  output keys : ['loc', 'scale']
  loc shape   : torch.Size([4, 4])
  scale shape : torch.Size([4, 4])


In [11]:
# Same dict parametrization, batch form: returns a list of independent dict CPDs
normal_cpds = ParametricCPD(
    concepts=['z1', 'z2', 'z3'],
    parametrization={'loc': nn.Linear(32, 4), 'scale': nn.Linear(32, 4)},
    parents=[input_var],
)
print(type(normal_cpds), len(normal_cpds))
for cpd in normal_cpds:
    print(' ', cpd.concept)


<class 'list'> 3
  z1
  z2
  z3


### 3.2 `in_features` and parent management

In [12]:
# in_features = sum of parent variable sizes
smoke_var = ConceptVariable(concept='smoke', distribution=Bernoulli, size=1)
tub_var   = ConceptVariable(concept='tub',   distribution=Bernoulli, size=1)

cpd_either = ParametricCPD(
    concept='either',
    parametrization=nn.Linear(2, 1),  # smoke(1) + tub(1)
    parents=[smoke_var, tub_var],
)
print('either in_features:', cpd_either.in_features)  # 2


either in_features: 2


### 3.3 `forward()` and `sample()`

- **`cpd.forward(*args, **kwargs)`** — plain PyTorch forward; runs the parametrization. Returns a `Tensor` for single-module parametrizations and a `Dict[str, Tensor]` (keyed by parameter name) for `ModuleDict` parametrizations. Used by `ForwardInference` subclasses.
- **`cpd.sample(context, obs)`** — Pyro-aware; builds the distribution via `variable.make_distribution(...)` and calls `pyro.sample()` (or `pyro.deterministic()` for `Delta`). Used by `BayesianNetwork.forward()`. *(Renamed from `pyro_forward` in earlier versions.)*


In [13]:
# Plain forward (no Pyro)
x = torch.randn(4, 32)
logits = cpd_asia(x)  # calls cpd_asia.parametrization.forward(x)
print('plain forward output:', logits.shape)  # (4, 1) — raw logit

plain forward output: torch.Size([4, 1])


---
## 4. BayesianNetwork / ProbabilisticModel

**File:** `torch_concepts/nn/modules/mid/models/probabilistic_model.py`

<style>
.bn-wrap { font-family: 'SF Mono', 'Fira Code', 'Consolas', monospace; max-width: 700px; padding: 8px 0; display: flex; flex-direction: column; gap: 10px; }
.bn-card { background: #f6f4ff; border: 1px solid #c5baff; border-radius: 10px; overflow: hidden; }
.bn-card-header { background: #e6e1ff; padding: 10px 14px 8px; display: flex; align-items: baseline; gap: 10px; border-bottom: 1px solid #c9c0ff; }
.bn-badge { font-size: 11px; font-weight: 700; background: #7c6bff; color: #fff; padding: 2px 9px; border-radius: 20px; white-space: nowrap; }
.bn-alias { font-size: 10.5px; color: #7c6bff; opacity: .7; font-style: italic; }
.bn-role { font-size: 11px; color: #3a28aa; opacity: .6; margin-left: auto; }
.bn-steps { list-style: none; margin: 0; padding: 10px 14px 12px; display: flex; flex-direction: column; gap: 5px; }
.bn-step { display: flex; align-items: flex-start; gap: 9px; }
.step-num { font-size: 10px; font-weight: 700; background: #7c6bff; color: #fff; border-radius: 50%; width: 16px; height: 16px; display: flex; align-items: center; justify-content: center; flex-shrink: 0; margin-top: 1px; }
.step-text { font-size: 11.5px; color: #3a28aa; line-height: 1.55; }
.step-text code { background: #e6e1ff; border-radius: 3px; padding: 0 4px; font-size: 11px; }
.hier-card { background: #f0eeff; border: 1px solid #c5baff; border-radius: 10px; overflow: hidden; }
.hier-header { background: #e6e1ff; padding: 8px 14px; border-bottom: 1px solid #c9c0ff; }
.hier-title { font-size: 10px; letter-spacing: .08em; text-transform: uppercase; color: #3a28aa; opacity: .6; }
.hier-body { padding: 12px 14px; }
.h-row { display: flex; align-items: center; gap: 6px; padding: 3px 0; }
.h-indent1 { padding-left: 18px; } .h-indent2 { padding-left: 36px; }
.h-connector { font-size: 13px; color: #9989e8; opacity: .6; margin-right: 2px; }
.h-node { font-size: 12px; padding: 3px 10px; border-radius: 6px; border: 1px solid; white-space: nowrap; }
.h-node.root     { background: #ddd8ff; border-color: #a99cff; color: #2a1a90; font-weight: 700; }
.h-node.mid      { background: #e8e4ff; border-color: #c0b5ff; color: #3a28aa; font-weight: 600; }
.h-node.concrete { background: #7c6bff; border-color: #6458e0; color: #fff; font-weight: 700; }
.h-note { font-size: 10px; color: #7c6bff; opacity: .6; margin-left: 6px; font-style: italic; }
.h-alias-row { display: flex; align-items: center; gap: 6px; margin-top: 6px; padding-left: 54px; }
.h-alias-pill { font-size: 10.5px; padding: 2px 8px; border-radius: 20px; background: #fff; border: 1px dashed #9989e8; color: #5040b0; }
.h-alias-label { font-size: 10px; color: #7c6bff; opacity: .55; font-style: italic; }
.planned-card { background: #f8f8f6; border: 1px dashed #bbb9b0; border-radius: 10px; padding: 9px 14px; display: flex; align-items: center; gap: 10px; }
.planned-label { font-size: 10px; font-weight: 700; letter-spacing: .07em; text-transform: uppercase; color: #888780; background: #e8e6e0; border-radius: 4px; padding: 2px 7px; white-space: nowrap; }
.planned-pill { font-size: 11px; padding: 2px 9px; border-radius: 20px; background: #fff; border: 1px dashed #bbb9b0; color: #666460; }
</style>
<div class="bn-wrap">
  <div class="bn-card">
    <div class="bn-card-header">
      <span class="bn-badge">BayesianNetwork</span>
      <span class="bn-alias">aka ProbabilisticModel (backward compat)</span>
      <span class="bn-role">DAG container</span>
    </div>
    <ul class="bn-steps">
      <li class="bn-step"><span class="step-num">1</span><span class="step-text">Registers all <code>ParametricCPD</code> factors in an <code>nn.ModuleDict</code></span></li>
      <li class="bn-step"><span class="step-num">2</span><span class="step-text">Resolves string parent references → <code>Variable</code> objects</span></li>
      <li class="bn-step"><span class="step-num">3</span><span class="step-text">Computes a topological order <span style="opacity:.55">(Kahn's algorithm)</span></span></li>
      <li class="bn-step"><span class="step-num">4</span><span class="step-text">Provides <code>forward()</code> as a full Pyro generative program</span></li>
      <li class="bn-step"><span class="step-num">5</span><span class="step-text">Provides <code>query()</code> for posterior-predictive sampling</span></li>
    </ul>
  </div>
  <div class="hier-card">
    <div class="hier-header"><span class="hier-title">§ 4.1 — Class hierarchy</span></div>
    <div class="hier-body">
      <div class="h-row"><span class="h-node root">PyroModule</span></div>
      <div class="h-row h-indent1"><span class="h-connector">└─</span><span class="h-node mid">_ProbabilisticModelBase</span><span class="h-note">abstract · shared infra: lookup, parent-building</span></div>
      <div class="h-row h-indent2"><span class="h-connector">└─</span><span class="h-node concrete">BayesianNetwork</span><span class="h-note">concrete directed PGM</span></div>
      <div class="h-alias-row"><span style="font-size:11px;color:#9989e8;">↑</span><span class="h-alias-pill">ProbabilisticModel</span><span class="h-alias-label">alias</span></div>
    </div>
  </div>
  <div class="planned-card">
    <span class="planned-label">Planned</span>
    <span style="font-size:10.5px;color:#888780;font-style:italic;margin-right:4px;">not yet implemented</span>
    <span class="planned-pill">MarkovRandomField</span>
    <span class="planned-pill">ChainGraph</span>
  </div>
</div>


In [14]:
from torch_concepts.nn.modules.mid.models.probabilistic_model import BayesianNetwork, ProbabilisticModel

# Minimal 3-node DAG:  input (Delta/32) → smoke (Bernoulli) → bronc (Bernoulli)
pyro.clear_param_store()

inp_var   = LatentVariable(concept='input', distribution=Delta,    size=8)
smk_var   = ConceptVariable(concept='smoke', distribution=Bernoulli, size=1)
brc_var   = ConceptVariable(concept='bronc', distribution=Bernoulli, size=1)

cpd_inp   = ParametricCPD(concept='input', parametrization=nn.Identity())
cpd_smk   = ParametricCPD(concept='smoke', parametrization=nn.Linear(8, 1),  parents=[inp_var])
cpd_brc   = ParametricCPD(concept='bronc', parametrization=nn.Linear(1, 1),  parents=[smk_var])

pgm = BayesianNetwork(
    variables=[inp_var, smk_var, brc_var],
    factors=[cpd_inp, cpd_smk, cpd_brc],
)
print(pgm)


BayesianNetwork(
  (factors): ModuleDict(
    (input): ParametricCPD(concept='input', parametrization=Identity, parents=[])
    (smoke): ParametricCPD(concept='smoke', parametrization=Linear, parents=['input'])
    (bronc): ParametricCPD(concept='bronc', parametrization=Linear, parents=['smoke'])
  )
)


In [15]:
# Inspect the registered state after construction
print('factors (ModuleDict keys):', list(pgm.factors.keys()))
print('sorted_variables:', [v.concept for v in pgm.sorted_variables])
print('concept_to_variable:', list(pgm.concept_to_variable.keys()))

# Parent resolution
print('\nbronc parents:', [p.concept for p in pgm.get_variable_parents('bronc')])

factors (ModuleDict keys): ['input', 'smoke', 'bronc']
sorted_variables: ['input', 'smoke', 'bronc']
concept_to_variable: ['input', 'smoke', 'bronc']

bronc parents: ['smoke']


### 4.2 `_build_parent_kwargs` — the input concatenation logic

This **static method** (on `_ProbabilisticModelBase`) is the central orchestrator for feeding parent values to a CPD. It handles three calling conventions:

1. **Standard positional** — concatenates all parent tensors along the last dim and passes as the first positional arg
2. **PyC-style keyword args** — if the CPD's `forward` signature contains `concepts`, `latent`, or `exogenous` kwargs, parents are split by type
3. **Root nodes** — evidence value passed directly through `cpd(raw)`

`ConceptVariable` parents go into `parent_concepts`; `LatentVariable`/`ExogenousVariable` parents go into `parent_input`.

In [16]:
from torch_concepts.nn.modules.mid.models.probabilistic_model import _ProbabilisticModelBase

B = 4
# Build a fake context as would exist during a forward pass
context = {
    'input': torch.randn(B, 8),
    'smoke': torch.sigmoid(torch.randn(B, 1)),  # already activated probability
}
evidence = {'input': context['input']}

# For bronc (parent = smoke, a ConceptVariable)
cpd_bronc = pgm.get_module_of_concept('bronc')
kwargs = _ProbabilisticModelBase._build_parent_kwargs(cpd_bronc, context)
print('kwargs keys :', list(kwargs.keys()))
print('input shape :', list(kwargs.values())[0].shape, '\n      values : ', list(kwargs.values())[0])  # (B, 1)

kwargs keys : ['input']
input shape : torch.Size([4, 1]) 
      values :  tensor([[0.3394],
        [0.4324],
        [0.3936],
        [0.2145]])


### 4.3 `forward()` — the Pyro generative model

This is the heart of the Pyro integration. It runs each variable in topological order inside a `pyro.plate('data', batch_size)` context:

```python
for var in self.sorted_variables:
    params = self._run_cpd(cpd, context, obs_dict)
    if var.is_deterministic:
        context[name] = pyro.deterministic(name, params)   # Delta → no sample site
    else:
        d = var.make_distribution(params)
        obs = obs_dict.get(name)                           # None = latent, tensor = observed
        context[name] = pyro.sample(name, d, obs=obs)
```

This design means:
- **Training with observed concepts** → pass `targets={'smoke': c_batch}` → `obs=c_batch` → log p(c|parents) accumulated in ELBO
- **Latent variables** → `obs=None` → sampled from prior during model; sampled from guide during ELBO estimation

In [17]:
import pyro.poutine as poutine

B = 4
x = torch.randn(B, 8)

pgm.eval()
with torch.no_grad():
    # Trace the generative model to see every Pyro sample site
    tr = poutine.trace(pgm).get_trace({'input': x})

print('Pyro trace sites:')
for name, node in tr.nodes.items():
    if node['type'] == 'sample':
        print(f'  [{name}]  value.shape={node["value"].shape}  observed={node.get("is_observed", False)}')
    elif node['type'] == 'deterministic':
        print(f'  [{name}]  (deterministic)  value.shape={node["value"].shape}')

Pyro trace sites:
  [data]  value.shape=torch.Size([4])  observed=False
  [input]  value.shape=torch.Size([4, 8])  observed=True
  [smoke]  value.shape=torch.Size([4, 1])  observed=False
  [bronc]  value.shape=torch.Size([4, 1])  observed=False


### 4.4 `query()` — posterior-predictive API

`pgm.query()` provides a unified posterior-predictive interface. The `method` argument selects the inference algorithm:

| `method` | Algorithm | Best for |
|---|---|---|
| `None` / `"ancestral"` | `Predictive` (prior or guide) | Forward queries |
| `"importance"` | `ImportanceQuery` (IS resampling) | Backward / v-structure queries |
| `"nuts"` / `"hmc"` | `MCMCQuery` | Continuous latents, exact posterior |
| `"exact_discrete"` | `ExactDiscreteQuery` (variable elimination) | Discrete latents, exact posterior |

In [18]:
with torch.no_grad():
    samples = pgm.query(
        variables=['smoke', 'bronc'],
        evidence={'input': x},
        num_samples=50,
        method=None,   # ancestral (prior predictive)
    )

print('smoke samples shape :', samples.samples['smoke'].shape)   # (50, B, 1)
print('bronc samples shape :', samples.samples['bronc'].shape)

smoke samples shape : torch.Size([50, 4, 1])
bronc samples shape : torch.Size([50, 4, 1])


---
## 5. ForwardInference (abstract base)

**File:** `torch_concepts/nn/modules/mid/inference/forward.py`

`ForwardInference` is the abstract base for all non-Pyro-loop inference engines. It implements the topological forward sweep and leaves `activate()` abstract for subclasses.

### 5.1 Initialization

On construction, `ForwardInference`:
1. Runs `_topological_sort()` → `sorted_variables` + `levels` (list of lists)
2. Caches CPD signatures and parent lists to avoid `inspect.signature()` overhead per batch
3. Detects shared CPDs

### 5.2 Topological levels for parallelism

Variables at the same depth (no inter-dependencies within a level) are computed in parallel:
- CPU parallelism via `ThreadPoolExecutor`
- GPU parallelism via CUDA streams

In [19]:
from torch_concepts.nn.modules.mid.inference.deterministic import DeterministicInference

engine = DeterministicInference(pgm)

print('Sorted variables:', [v.concept for v in engine.sorted_variables])
print('Levels (parallelism groups):')
for i, level in enumerate(engine.levels):
    print(f'  Level {i}: {[v.concept for v in level]}')

Sorted variables: ['input', 'smoke', 'bronc']
Levels (parallelism groups):
  Level 0: ['input']
  Level 1: ['smoke']
  Level 2: ['bronc']


### 5.3 The `query()` method flow

<style>
.qf-wrap { font-family: 'SF Mono','Fira Code','Consolas',monospace; max-width:700px; padding:8px 0; }
.qf-card { background:#f6f4ff; border:1px solid #c5baff; border-radius:10px; overflow:hidden; }
.qf-header { background:#e6e1ff; border-bottom:1px solid #c9c0ff; padding:10px 14px 8px; display:flex; align-items:baseline; gap:8px; }
.qf-fn { font-size:12px; font-weight:700; color:#3a28aa; }
.qf-fn span { color:#7c6bff; }
.trunk { display:flex; gap:0; }
.trunk-line { width:2px; background:#c5baff; border-radius:2px; margin:0 10px 0 6px; flex-shrink:0; }
.trunk-items { display:flex; flex-direction:column; gap:6px; flex:1; }
.br { display:flex; align-items:flex-start; gap:0; }
.br-hook { display:flex; flex-direction:column; align-items:center; width:18px; flex-shrink:0; margin-right:6px; }
.br-h-line { height:12px; width:2px; background:#c5baff; }
.br-elbow { width:14px; height:10px; border-left:2px solid #c5baff; border-bottom:2px solid #c5baff; border-radius:0 0 0 4px; margin-top:-1px; }
.node { display:inline-flex; align-items:center; gap:6px; padding:4px 10px; border-radius:6px; border:1px solid; font-size:11.5px; line-height:1.4; flex-wrap:wrap; }
.node.step     { background:#ede9ff; border-color:#c5baff; color:#3a28aa; }
.node.loop     { background:#e6e1ff; border-color:#b0a3ff; color:#3020a0; font-weight:600; }
.node.call     { background:#7c6bff; border-color:#6458e0; color:#fff; font-weight:700; }
.node.leaf     { background:#fff; border-color:#c5baff; color:#4a38b0; }
.node.abstract { background:#fff; border-color:#9989e8; color:#5040b0; font-style:italic; }
.node.output   { background:#1d9e75; border-color:#0f6e56; color:#fff; font-weight:700; }
.node.mix      { background:#fff9ee; border-color:#fac775; color:#7a4900; }
.node code     { background:rgba(0,0,0,.08); border-radius:3px; padding:0 4px; font-size:10.5px; font-style:normal; }
.tag { font-size:9.5px; padding:1px 5px; border-radius:3px; font-weight:700; letter-spacing:.05em; white-space:nowrap; }
.tag.abs      { background:#9989e8; color:#fff; }
.tag.opt      { background:#d08000; color:#fff; }
.tag.loop-tag { background:#6458e0; color:#fff; }
.sub-indent { padding-left:28px; }
</style>
<div class="qf-wrap">
<div class="qf-card">
  <div class="qf-header">
    <span class="qf-fn"><span>query</span>(concept_names, evidence, ground_truth=None, return_parameters=False, return_probs=True, …)</span>
  </div>
  <div style="padding:12px 14px 14px;display:flex;flex-direction:column;gap:0;">
    <div class="trunk">
      <div class="trunk-line"></div>
      <div class="trunk-items">
        <div class="br">
          <div class="br-hook"><div class="br-h-line"></div><div class="br-elbow"></div></div>
          <div class="node step">build lazy ancestor set <span class="tag opt">if lazy=True</span></div>
        </div>
        <div class="br">
          <div class="br-hook"><div class="br-h-line"></div><div class="br-elbow"></div></div>
          <div style="display:flex;flex-direction:column;gap:5px;flex:1;">
            <div class="node loop"><span class="tag loop-tag">for each</span> level in topological order</div>
            <div class="sub-indent" style="display:flex;flex-direction:column;gap:4px;">
              <div class="node call">_predict_level(level, evidence, results)</div>
              <div class="sub-indent" style="display:flex;flex-direction:column;gap:4px;">
                <div class="node loop"><span class="tag loop-tag">for each</span> CPD in level <span style="opacity:.5;font-size:10px;font-weight:400;">(deduplicated)</span></div>
                <div class="sub-indent" style="display:flex;flex-direction:column;gap:4px;">
                  <div class="node call">_compute_single_variable(var, evidence, results)</div>
                  <div class="sub-indent" style="display:flex;flex-direction:column;gap:3px;">
                    <div class="node leaf"><code>root</code> cpd(evidence[concept]) → raw params</div>
                    <div class="node leaf"><code>child</code> cpd(**build_parent_kwargs()) → raw params</div>
                  </div>
                  <div class="node abstract">activate(raw_params, variable) <span class="tag abs">ABSTRACT</span></div>
                  <div class="node mix">optionally mix with GT via <code>p</code> parameter <span class="tag opt">optional</span></div>
                </div>
              </div>
            </div>
          </div>
        </div>
        <div class="br">
          <div class="br-hook"><div class="br-h-line"></div><div class="br-elbow"></div></div>
          <div style="display:flex;align-items:center;gap:7px;flex-wrap:wrap;">
            <div class="node step">concatenate results for queried concepts</div>
            <span style="font-size:13px;color:#9989e8;">→</span>
            <div class="node output">InferenceOutput</div>
          </div>
        </div>
      </div>
    </div>
  </div>
</div>
</div>

> The previous `return_logits` flag has been renamed to `return_parameters`, since the field stores raw distribution parameters (logits, loc/scale, etc.) — not strictly logits.

### 5.4 The `p` parameter (teacher forcing)

- `p=0.0` (default): standard forward inference — model predictions propagated
- `p=1.0`: ground truth always propagated (independent training, like CEM)
- `0 < p < 1`: stochastic mix per-sample (Bernoulli mask over the batch)


---
## 6. DeterministicInference

**File:** `torch_concepts/nn/modules/mid/inference/deterministic.py`

Implements `activate()` by building the variable's distribution via `variable.make_distribution(raw)` and returning its analytical mean (`dist.mean`). For relaxed distributions that have no analytical mean (e.g. `RelaxedBernoulli`, `RelaxedOneHotCategorical`), it falls back to `sigmoid` / `softmax` of the underlying logits. No sampling occurs — this is the standard maximum-likelihood forward pass.


In [20]:
from torch_concepts.nn.modules.mid.inference.deterministic import DeterministicInference

det_engine = DeterministicInference(pgm)

B = 8
x = torch.randn(B, 8)

result = det_engine.query(
    query=['smoke', 'bronc'],
    evidence={'input': x},
    return_probs=True,
    return_parameters=True,
)

print('InferenceOutput fields:')
print('  parameters.shape :', result.parameters.shape)  # (B, 2) — concatenated raw params
print('  probs.shape      :', result.probs.shape)       # (B, 2) — activated
print('  loss             :', result.loss)              # None (no internal loss)

# probs are always in [0,1]; parameters are raw
print('\nSmoke probs (first 3):', result.probs[:3, 0].tolist())


InferenceOutput fields:
  parameters.shape : torch.Size([8, 2])
  probs.shape      : torch.Size([8, 2])
  loss             : None

Smoke probs (first 3): [0.5186192393302917, 0.4933742880821228, 0.37399712204933167]


In [21]:
# Training loop pattern with DeterministicInference
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(pgm.parameters(), lr=1e-3)

pgm.train()
for step in range(5):
    x_b = torch.randn(16, 8)
    c_b = torch.randint(0, 2, (16, 2)).float()  # fake labels: [smoke, bronc]

    optimizer.zero_grad()
    result = det_engine.query(
        query=['smoke', 'bronc'],
        evidence={'input': x_b},
        return_parameters=True,
        return_probs=False,
    )
    # For Bernoulli concepts the raw `parameters` field stores the logit,
    # so BCE-with-logits is the natural loss.
    loss = loss_fn(result.parameters, c_b)
    loss.backward()
    optimizer.step()
    if step % 1 == 0:
        print(f'  step {step+1}: loss={loss.item():.4f}')

pgm.eval()


  step 1: loss=0.6316
  step 2: loss=0.6818
  step 3: loss=0.6902
  step 4: loss=0.6830
  step 5: loss=0.7454


BayesianNetwork(
  (factors): ModuleDict(
    (input): ParametricCPD(concept='input', parametrization=Identity, parents=[])
    (smoke): ParametricCPD(concept='smoke', parametrization=Linear, parents=['input'])
    (bronc): ParametricCPD(concept='bronc', parametrization=Linear, parents=['smoke'])
  )
)

### 6.1 `ground_truth_to_evidence()`

Used when `p > 0` (teacher forcing). Converts discrete GT tensors to the activated representation:
- Binary: `(B,)` or `(B, 1)` of 0/1 → `(B, 1)` float
- Categorical: class indices → one-hot `(B, k)`

---
## 7. AncestralSamplingInference

**File:** `torch_concepts/nn/modules/mid/inference/ancestral.py`

Implements `activate()` as stochastic sampling from the distribution parameterized by the CPD output. Differences from `DeterministicInference`:

- Uses `.rsample()` when the distribution supports reparameterization; falls back to `.sample()`
- Passes `logits=` or `probs=` based on the `log_probs` flag
- Uses `make_distribution()` for multi-parameter distributions (Normal, MultivariateNormal)
- Caches distribution signatures at init time to avoid per-batch introspection

In [22]:
from torch_concepts.nn.modules.mid.inference.ancestral import AncestralSamplingInference

anc_engine = AncestralSamplingInference(pgm)

x = torch.randn(8, 8)
result1 = anc_engine.query(['smoke', 'bronc'], evidence={'input': x}, return_probs=True)
result2 = anc_engine.query(['smoke', 'bronc'], evidence={'input': x}, return_probs=True)

print('Ancestral result 1 (smoke, first 3):',  result1.probs[:3, 0].tolist())
print('Ancestral result 2 (smoke, first 3):',  result2.probs[:3, 0].tolist())
print('Are identical?', torch.equal(result1.probs, result2.probs))  # False — stochastic

Ancestral result 1 (smoke, first 3): [1.0, 0.0, 1.0]
Ancestral result 2 (smoke, first 3): [0.0, 1.0, 1.0]
Are identical? False


In [23]:
# For Bernoulli variables, samples are in {0, 1} (discrete) or [0,1] (relaxed)
print('Unique smoke values (Bernoulli → discrete 0/1):', result1.probs[:, 0].unique().tolist())

Unique smoke values (Bernoulli → discrete 0/1): [0.0, 1.0]


---
## 8. IndependentInference

**File:** `torch_concepts/nn/modules/mid/inference/independent.py`

A thin subclass of `DeterministicInference` that forces `p=1.0`, meaning ground truth concept labels are **always** propagated to downstream predictors during training. This implements the "independent" or CEM-style training where each concept predictor is trained independently from child predictors.

Equivalent to `DeterministicInference(pgm, p=1.0)`.

In [24]:
from torch_concepts.nn.modules.mid.inference.independent import IndependentInference

ind_engine = IndependentInference(pgm)
print('p =', ind_engine.p)  # 1.0

p = 1.0


---
## 9. ELBOInference & AmortizedGuide

### 9.1 ELBOInference

**File:** `torch_concepts/nn/modules/mid/inference/elbo.py`

`ELBOInference` wraps Pyro's `ELBOModule` (a differentiable ELBO estimator) and exposes a `query()` method returning an `InferenceOutput` with:
- `result.loss` — negative ELBO (differentiable; call `.backward()` in the training loop)
- `result.probs` / `result.parameters` — posterior-predictive marginals from the guide (no gradient, for monitoring)

### 9.2 AmortizedGuide

**File:** `torch_concepts/nn/modules/mid/inference/guide.py`

Automatically built from the PGM's CPD networks. For each non-Delta, non-observed variable:
1. Collects parent values from a running context (topological order)
2. Runs the **same** `ParametricCPD.parametrization` as the model → shared weights
3. Calls `pyro.sample(name, dist)` (without `obs`)

The guide's `encoders` `nn.ModuleDict` holds references to the exact same `nn.Module` objects as the model's CPDs — **weights are shared between model and guide**.


In [25]:
from torch_concepts.nn.modules.mid.inference.elbo import ELBOInference
from torch_concepts.nn.modules.mid.inference.guide import AmortizedGuide

pyro.clear_param_store()

elbo_engine = ELBOInference(pgm, num_particles=4)

# The ELBO module wraps model + guide; its .parameters() includes both
print('ELBOInference internal structure:')
print('  elbo_module type  :', type(elbo_engine.elbo_module).__name__)
print('  model type        :', type(elbo_engine.elbo_module.model).__name__)
print('  guide type        :', type(elbo_engine.elbo_module.guide).__name__)

guide = elbo_engine.elbo_module.guide
print('  guide encoders    :', list(guide.encoders.keys()))

ELBOInference internal structure:
  elbo_module type  : ELBOModule
  model type        : BayesianNetwork
  guide type        : AmortizedGuide
  guide encoders    : ['input', 'smoke', 'bronc']


In [26]:
# Training loop pattern with ELBOInference
optimizer = torch.optim.Adam(elbo_engine.parameters(), lr=1e-3)

pgm.train()
for step in range(3):
    x_b = torch.randn(16, 8)
    c_b = torch.randint(0, 2, (16, 2)).float()  # fake labels

    optimizer.zero_grad()
    result = elbo_engine.query(
        query=['smoke', 'bronc'],
        evidence={'input': x_b},
        targets={'smoke': c_b[:, :1], 'bronc': c_b[:, 1:]},  # observed targets
    )
    result.loss.backward()  # negative ELBO
    optimizer.step()
    print(f'  step {step}: neg-ELBO={result.loss.item():.4f}')

pgm.eval()

  step 0: neg-ELBO=23.4123
  step 1: neg-ELBO=24.3854
  step 2: neg-ELBO=21.3401


BayesianNetwork(
  (factors): ModuleDict(
    (input): ParametricCPD(concept='input', parametrization=Identity, parents=[])
    (smoke): ParametricCPD(concept='smoke', parametrization=Linear, parents=['input'])
    (bronc): ParametricCPD(concept='bronc', parametrization=Linear, parents=['smoke'])
  )
)

### 9.3 Guide parameter sharing (critical for correctness)

Because the `AmortizedGuide` holds references to the **same** `nn.Module` instances as the `BayesianNetwork.factors`, the optimizer updating `elbo_engine.parameters()` trains both model and guide simultaneously through a single set of weights.

In [27]:
# Verify shared weights between model and guide
model_smoke_linear = pgm.factors['smoke'].parametrization
guide_smoke_encoder = guide.encoders['smoke']

print('Model smoke parametrization id :', id(model_smoke_linear))
print('Guide smoke encoder id          :', id(guide_smoke_encoder))
print('Are the same object?            :', model_smoke_linear is guide_smoke_encoder)

Model smoke parametrization id : 138821760549232
Guide smoke encoder id          : 138821760549232
Are the same object?            : True


---
## 10. Posterior Query Engines

These three engines perform **inference about hidden or cause variables** given observations. They differ in exactness, scalability, and the types of variables they support.

### 10.1 ImportanceQuery

**File:** `torch_concepts/nn/modules/mid/inference/importance.py`

Uses Pyro's `Importance` sampler:
1. Draw `num_samples` proposals from the prior (or a guide)
2. Weight by unnormalized importance weights: `w_i ∝ p(evidence | z_i) / q(z_i)`
3. Resample with replacement proportionally to `w_i`

Works for any evidence pattern including backward queries (child → parent).

In [28]:
from torch_concepts.nn.modules.mid.inference.importance import ImportanceQuery

imp_query = ImportanceQuery(pgm)

x_single = torch.randn(1, 8)

# Forward query
fwd_samples = imp_query.query(
    variables=['smoke', 'bronc'],
    evidence={'input': x_single},
    num_samples=200,
)
print('Forward IS samples:')
print('  smoke shape:', fwd_samples.samples['smoke'].shape)   # (200, 1, 1)
print('  P(smoke=1) ≈', fwd_samples.samples['smoke'].float().mean().item())

Forward IS samples:
  smoke shape: torch.Size([200, 1, 1])
  P(smoke=1) ≈ 0.4350000023841858


In [29]:
# Backward query: condition on bronc=1, infer P(smoke | input, bronc=1)
back_samples = imp_query.query(
    variables=['smoke'],
    evidence={'input': x_single, 'bronc': torch.ones(1, 1)},
    num_samples=300,
)
print('Backward IS — P(smoke=1 | bronc=1) ≈', back_samples.samples['smoke'].float().mean().item())

Backward IS — P(smoke=1 | bronc=1) ≈ 0.4266666769981384


### 10.2 ExactDiscreteQuery

**File:** `torch_concepts/nn/modules/mid/inference/exact_discrete.py`

Uses Pyro's `infer_discrete` (variable elimination via `config_enumerate`):
- Exact posterior over **discrete** latents — no approximation error
- Works for both forward and backward queries
- Draws `num_samples` independent exact-posterior samples
- `temperature=1` → sampling; `temperature=0` → MAP

> **Not applicable for continuous latents** — use `MCMCQuery` instead.

In [30]:
from torch_concepts.nn.modules.mid.inference.exact_discrete import ExactDiscreteQuery

exact_query = ExactDiscreteQuery(pgm, temperature=1)

# Exact backward query: P(smoke | input, bronc=1)
exact_samples = exact_query.query(
    variables=['smoke'],
    evidence={'input': x_single, 'bronc': torch.ones(1, 1)},
    num_samples=200,
)
if 'smoke' in exact_samples.samples:
    print('Exact — P(smoke=1 | bronc=1) ≈', exact_samples.samples['smoke'].float().mean().item())

Exact — P(smoke=1 | bronc=1) ≈ 0.3499999940395355


### 10.3 MCMCQuery

**File:** `torch_concepts/nn/modules/mid/inference/mcmc.py`

Uses Pyro's NUTS or HMC:
- Asymptotically exact for **continuous** latents
- Has a warmup (burn-in) phase
- Multi-chain support
- **Not applicable for discrete variables** (NUTS/HMC require continuous latents)

In [31]:
# MCMCQuery example: only run if you have a model with continuous latents
# (Bernoulli is discrete, so NUTS is not applicable here)
# Shown for illustration — would be used with Normal-distributed latents

from torch_concepts.nn.modules.mid.inference.mcmc import MCMCQuery

print('MCMCQuery parameters:')
mcmc_q = MCMCQuery(pgm, kernel='nuts', warmup_steps=50, num_chains=1)
print('  kernel    :', mcmc_q._kernel_name)
print('  warmup    :', mcmc_q._warmup_steps)
print('  num_chains:', mcmc_q._num_chains)
# mcmc_q.query(variables, evidence, num_samples)  # Would call NUTS on continuous latents

MCMCQuery parameters:
  kernel    : nuts
  warmup    : 50
  num_chains: 1


### 10.4 Choosing an inference algorithm

| Task | Variables | Recommended engine |
|---|---|---|
| Training (gradient) | Any | `DeterministicInference` or `ELBOInference` |
| Forward prediction | Any | `DeterministicInference` or `AncestralSamplingInference` |
| Backward / causal query (discrete) | Bernoulli, Categorical | `ExactDiscreteQuery` (exact) or `ImportanceQuery` (approximate) |
| Backward / causal query (continuous) | Normal | `MCMCQuery` (exact) or `ImportanceQuery` (approximate) |
| Independent training | Any | `IndependentInference` |

---
## 11. InferenceOutput

**File:** `torch_concepts/nn/modules/outputs.py`

All `ForwardInference` subclasses and `ELBOInference.query()` return an `InferenceOutput` dataclass:

```python
@dataclass
class InferenceOutput:
    parameters: Optional[torch.Tensor] = None  # raw CPD output (logits/loc/scale/...), (B, sum_of_sizes)
    probs:      Optional[torch.Tensor] = None  # activated, (B, sum_of_sizes)
    joint:      Optional[torch.Tensor] = None  # unnorm. log joint (rarely used)
    loss:       Optional[torch.Tensor] = None  # only set by ELBOInference
    samples:    Optional[Dict[str, Tensor]] = None  # set by sample-based posterior engines
```

For `ForwardInference` engines, the tensors have shape `(B, sum_of_queried_concept_sizes)` — concepts are concatenated in the order they appear in the `query` argument.

> The `parameters` field replaces the previous `logits` field, since for non-Bernoulli/Categorical distributions the raw values are not strictly logits.

Sample-based posterior engines (`ImportanceQuery`, `ExactDiscreteQuery`, `MCMCQuery`, and `BayesianNetwork.query()`) also return `InferenceOutput`, but populate `.samples` (a `Dict[str, Tensor(num_samples, *batch, size)]`) instead of `.parameters`. Their `.probs` is the empirical mean over the leading sample dimension, concatenated in the requested-variable order.


In [32]:
from torch_concepts.nn.modules.outputs import InferenceOutput

result = det_engine.query(
    query=['smoke', 'bronc'],
    evidence={'input': torch.randn(8, 8)},
    return_parameters=True,
    return_probs=True,
)
print(type(result))
print('parameters:', result.parameters.shape)  # (8, 2)
print('probs     :', result.probs.shape)       # (8, 2)
print('loss      :', result.loss)              # None

# Unpack per-concept
smoke_probs = result.probs[:, 0:1]
bronc_probs = result.probs[:, 1:2]
print('smoke probs shape:', smoke_probs.shape)


<class 'torch_concepts.nn.modules.outputs.InferenceOutput'>
parameters: torch.Size([8, 2])
probs     : torch.Size([8, 2])
loss      : None
smoke probs shape: torch.Size([8, 1])


---
## 12. End-to-End Example: Asia Bayesian Network

Reproduces the 8-node Asia BN from the attached example script in a self-contained, annotated form. 

<style>
.dag-wrap { font-family: 'SF Mono','Fira Code','Consolas',monospace; max-width:700px; padding:8px 0; }
.dag-card { background:#f6f4ff; border:1px solid #c5baff; border-radius:10px; overflow:hidden; }
.dag-header { background:#e6e1ff; border-bottom:1px solid #c9c0ff; padding:8px 14px; display:flex; align-items:center; gap:8px; }
.dag-label { font-size:10px; letter-spacing:.08em; text-transform:uppercase; color:#3a28aa; opacity:.6; }
.dag-sub { font-size:10px; color:#3a28aa; opacity:.4; margin-left:auto; }
.dag-body { padding:18px 20px 22px; }
svg.dag { width:100%; overflow:visible; }
.edge { stroke:#a99cff; stroke-width:1.8; fill:none; marker-end:url(#arr); }
</style>
<div class="dag-wrap">
<div class="dag-card">
  <div class="dag-header">
    <span class="dag-label">Asia Bayesian Network — DAG</span>
    <span class="dag-sub">32-dim embedding input</span>
  </div>
  <div class="dag-body">
<svg class="dag" viewBox="0 0 640 220" xmlns="http://www.w3.org/2000/svg">
  <defs>
    <marker id="arr" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto">
      <path d="M0,0 L0,6 L8,3 z" fill="#9989e8"/>
    </marker>
  </defs>
  <line class="edge" x1="88" y1="80" x2="158" y2="60"/>
  <line class="edge" x1="88" y1="100" x2="158" y2="145"/>
  <line class="edge" x1="222" y1="55" x2="292" y2="55"/>
  <line class="edge" x1="222" y1="145" x2="292" y2="115"/>
  <line class="edge" x1="222" y1="158" x2="292" y2="178"/>
  <line class="edge" x1="356" y1="55" x2="426" y2="85"/>
  <line class="edge" x1="356" y1="110" x2="426" y2="95"/>
  <line class="edge" x1="492" y1="90" x2="562" y2="70"/>
  <line class="edge" x1="492" y1="100" x2="562" y2="155"/>
  <line class="edge" x1="356" y1="178" x2="562" y2="165"/>
  <rect x="6" y="72" width="82" height="36" rx="7" fill="#7c6bff"/>
  <text x="47" y="86" text-anchor="middle" fill="#fff" font-weight="700" font-size="11" font-family="monospace">input</text>
  <text x="47" y="99" text-anchor="middle" fill="#ddd" font-size="9" font-family="monospace">(32-dim)</text>
  <rect x="160" y="38" width="62" height="30" rx="7" fill="#ede9ff" stroke="#c5baff"/>
  <text x="191" y="58" text-anchor="middle" fill="#3a28aa" font-weight="600" font-size="11" font-family="monospace">asia</text>
  <rect x="160" y="130" width="62" height="30" rx="7" fill="#ede9ff" stroke="#c5baff"/>
  <text x="191" y="150" text-anchor="middle" fill="#3a28aa" font-weight="600" font-size="11" font-family="monospace">smoke</text>
  <rect x="294" y="38" width="62" height="30" rx="7" fill="#ede9ff" stroke="#c5baff"/>
  <text x="325" y="58" text-anchor="middle" fill="#3a28aa" font-weight="600" font-size="11" font-family="monospace">tub</text>
  <rect x="294" y="96" width="62" height="30" rx="7" fill="#ede9ff" stroke="#c5baff"/>
  <text x="325" y="116" text-anchor="middle" fill="#3a28aa" font-weight="600" font-size="11" font-family="monospace">lung</text>
  <rect x="294" y="162" width="62" height="30" rx="7" fill="#ede9ff" stroke="#c5baff"/>
  <text x="325" y="182" text-anchor="middle" fill="#3a28aa" font-weight="600" font-size="11" font-family="monospace">bronc</text>
  <rect x="428" y="72" width="62" height="30" rx="7" fill="#ede9ff" stroke="#c5baff"/>
  <text x="459" y="92" text-anchor="middle" fill="#3a28aa" font-weight="600" font-size="11" font-family="monospace">either</text>
  <rect x="564" y="52" width="62" height="30" rx="7" fill="#1d9e75"/>
  <text x="595" y="72" text-anchor="middle" fill="#fff" font-weight="700" font-size="11" font-family="monospace">xray</text>
  <rect x="564" y="140" width="62" height="30" rx="7" fill="#1d9e75"/>
  <text x="595" y="160" text-anchor="middle" fill="#fff" font-weight="700" font-size="11" font-family="monospace">dysp</text>
</svg>
<div style="display:flex;gap:12px;margin-top:10px;padding-top:8px;border-top:1px solid #e0dbff;">
  <div style="display:flex;align-items:center;gap:5px;"><div style="width:14px;height:14px;border-radius:3px;background:#7c6bff;"></div><span style="font-size:10px;color:#3a28aa;opacity:.7;font-family:monospace;">embedding input</span></div>
  <div style="display:flex;align-items:center;gap:5px;"><div style="width:14px;height:14px;border-radius:3px;background:#ede9ff;border:1px solid #c5baff;"></div><span style="font-size:10px;color:#3a28aa;opacity:.7;font-family:monospace;">latent variable</span></div>
  <div style="display:flex;align-items:center;gap:5px;"><div style="width:14px;height:14px;border-radius:3px;background:#1d9e75;"></div><span style="font-size:10px;color:#3a28aa;opacity:.7;font-family:monospace;">observed / output</span></div>
</div>
  </div>
</div>
</div>

In [33]:
# --- Setup ---
from torch_concepts.nn.modules.mid.models.variable import ConceptVariable, LatentVariable
from torch_concepts.nn.modules.mid.models.cpd import ParametricCPD
from torch_concepts.nn.modules.mid.models.probabilistic_model import BayesianNetwork
from torch_concepts.distributions import Delta

EMB_DIM = 32

def build_asia_pgm():
    """Build a fresh Asia BN. Always call pyro.clear_param_store() before."""
    pyro.clear_param_store()

    # --- Variables ---
    input_var  = LatentVariable(concept='input',  distribution=Delta,    size=EMB_DIM)
    asia_var   = ConceptVariable(concept='asia',   distribution=Bernoulli, size=1)
    smoke_var  = ConceptVariable(concept='smoke',  distribution=Bernoulli, size=1)
    tub_var    = ConceptVariable(concept='tub',    distribution=Bernoulli, size=1)
    lung_var   = ConceptVariable(concept='lung',   distribution=Bernoulli, size=1)
    bronc_var  = ConceptVariable(concept='bronc',  distribution=Bernoulli, size=1)
    either_var = ConceptVariable(concept='either', distribution=Bernoulli, size=1)
    xray_var   = ConceptVariable(concept='xray',   distribution=Bernoulli, size=1)
    dysp_var   = ConceptVariable(concept='dysp',   distribution=Bernoulli, size=1)

    variables = [input_var, asia_var, smoke_var, tub_var, lung_var,
                 bronc_var, either_var, xray_var, dysp_var]

    # --- CPDs  (Asia DAG structure) ---
    cpd_input  = ParametricCPD(concept='input',  parametrization=nn.Identity())
    cpd_asia   = ParametricCPD(concept='asia',   parametrization=nn.Linear(EMB_DIM, 1), parents=[input_var])
    cpd_smoke  = ParametricCPD(concept='smoke',  parametrization=nn.Linear(EMB_DIM, 1), parents=[input_var])
    cpd_tub    = ParametricCPD(concept='tub',    parametrization=nn.Linear(1, 1),       parents=[asia_var])
    cpd_lung   = ParametricCPD(concept='lung',   parametrization=nn.Linear(1, 1),       parents=[smoke_var])
    cpd_bronc  = ParametricCPD(concept='bronc',  parametrization=nn.Linear(1, 1),       parents=[smoke_var])
    cpd_either = ParametricCPD(concept='either', parametrization=nn.Linear(2, 1),       parents=[lung_var, tub_var])
    cpd_xray   = ParametricCPD(concept='xray',   parametrization=nn.Linear(1, 1),       parents=[either_var])
    cpd_dysp   = ParametricCPD(concept='dysp',   parametrization=nn.Linear(2, 1),       parents=[either_var, bronc_var])

    factors = [cpd_input, cpd_asia, cpd_smoke, cpd_tub, cpd_lung,
               cpd_bronc, cpd_either, cpd_xray, cpd_dysp]

    return BayesianNetwork(variables=variables, factors=factors)


pgm_asia = build_asia_pgm()

print('Topological order:', [v.concept for v in pgm_asia.sorted_variables])
print('\nRegistered factors:')
for name, cpd in pgm_asia.factors.items():
    parents = [p.concept for p in cpd.parents]
    print(f'  {name:8s} ← {parents}')


Topological order: ['input', 'asia', 'smoke', 'tub', 'lung', 'bronc', 'either', 'xray', 'dysp']

Registered factors:
  input    ← []
  asia     ← ['input']
  smoke    ← ['input']
  tub      ← ['asia']
  lung     ← ['smoke']
  bronc    ← ['smoke']
  either   ← ['lung', 'tub']
  xray     ← ['either']
  dysp     ← ['either', 'bronc']


In [34]:
# --- Real Asia data via BnLearnDataModule (bnlearn) ---
from torch_concepts.data.datamodules.bnlearn import BnLearnDataModule

dm = BnLearnDataModule(name="asia", seed=42, n_gen=10_000, batch_size=512)
dm.setup("fit")

concept_names = dm.dataset.concept_names           # e.g. ['asia','tub','smoke',...]
CONCEPT_IDX   = {name: i for i, name in enumerate(concept_names)}
N_CONCEPTS    = len(concept_names)
EMB_DIM       = dm.dataset.input_data.shape[1]     # 32

print(f'  Concepts  : {concept_names}')
print(f'  Splits    : train={dm.train_len}, val={dm.val_len}, test={dm.test_len}')
print(f'  Embed dim : {EMB_DIM}')


INFO:torch_concepts.data.datasets.bnlearn:Loading dataset from /home/francesco/projects/pytorch_concepts/examples/utilization/1_pgm/data/asia
INFO:torch_concepts.data.base.datamodule:Input shape: (32,)
INFO:torch_concepts.data.base.datamodule:Using raw input data without backbone preprocessing.


  Concepts  : ['asia', 'tub', 'smoke', 'lung', 'bronc', 'either', 'xray', 'dysp']
  Splits    : train=7000, val=1000, test=2000
  Embed dim : 32


In [35]:
# --- Empirical marginals from the training split (used as ground truth for all comparisons) ---
train_concepts = dm.dataset.concepts[dm.trainset.indices].float()  # (Ntrain, 8)

# Unconditional: P(c_i = 1)
emp_marginals = train_concepts.mean(0)

# Conditional: P(concept | dysp=1, xray=1)  — for backward query B
mask_dysp1_xray1 = (train_concepts[:, CONCEPT_IDX['dysp']] == 1) & \
                   (train_concepts[:, CONCEPT_IDX['xray']] == 1)
C_back = train_concepts[mask_dysp1_xray1]
emp_smoke_given_dysp_xray = C_back[:, CONCEPT_IDX['smoke']].mean()
emp_bronc_given_dysp_xray = C_back[:, CONCEPT_IDX['bronc']].mean()

# Conditional: P(concept | xray=1)  — for hidden-variable query C
mask_xray1 = train_concepts[:, CONCEPT_IDX['xray']] == 1
C_xray = train_concepts[mask_xray1]
emp_either_given_xray = C_xray[:, CONCEPT_IDX['either']].mean()
emp_lung_given_xray   = C_xray[:, CONCEPT_IDX['lung']].mean()
emp_tub_given_xray    = C_xray[:, CONCEPT_IDX['tub']].mean()

print('Empirical unconditional marginals (training data):')
for name, m in zip(concept_names, emp_marginals.tolist()):
    print(f'  P({name:<6}) = {m:.3f}')

print(f'\nConditioning subset sizes:')
print(f'  |dysp=1, xray=1|  = {mask_dysp1_xray1.sum().item()} samples')
print(f'  |xray=1|           = {mask_xray1.sum().item()} samples')

print(f'\nEmpirical conditional marginals:')
print(f'  P(smoke  | dysp=1, xray=1) = {emp_smoke_given_dysp_xray:.3f}')
print(f'  P(bronc  | dysp=1, xray=1) = {emp_bronc_given_dysp_xray:.3f}')
print(f'  P(either | xray=1)         = {emp_either_given_xray:.3f}')
print(f'  P(lung   | xray=1)         = {emp_lung_given_xray:.3f}')
print(f'  P(tub    | xray=1)         = {emp_tub_given_xray:.3f}')


Empirical unconditional marginals (training data):
  P(asia  ) = 0.991
  P(tub   ) = 0.991
  P(smoke ) = 0.506
  P(lung  ) = 0.945
  P(bronc ) = 0.552
  P(either) = 0.936
  P(xray  ) = 0.891
  P(dysp  ) = 0.567

Conditioning subset sizes:
  |dysp=1, xray=1|  = 3698 samples
  |xray=1|           = 6240 samples

Empirical conditional marginals:
  P(smoke  | dysp=1, xray=1) = 0.618
  P(bronc  | dysp=1, xray=1) = 0.855
  P(either | xray=1)         = 0.999
  P(lung   | xray=1)         = 0.999
  P(tub    | xray=1)         = 1.000


### 12.1 Training with DeterministicInference (BCE loss)

In [36]:
pgm_asia = build_asia_pgm()
det_engine_asia = DeterministicInference(pgm_asia)
optimizer = torch.optim.Adam(pgm_asia.parameters(), lr=1e-3)
loss_fn   = nn.BCEWithLogitsLoss()

pgm_asia.train()
for epoch in range(200):
    epoch_loss = 0.0
    n_batches  = 0
    for batch in dm.train_dataloader():
        x_b = batch['inputs']['x'].cpu()        # (B, 32)
        c_b = batch['concepts']['c'].float()    # (B, 8)

        optimizer.zero_grad()
        result = det_engine_asia.query(
            query=concept_names,
            evidence={'input': x_b},
            return_parameters=True,
            return_probs=False,
        )
        loss = loss_fn(result.parameters, c_b)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        n_batches  += 1
    
    if (epoch + 1) % 20 == 0:
        print(f'Epoch {epoch+1:3d}  BCE = {epoch_loss / n_batches:.4f}')

pgm_asia.eval()
print('Training complete.')


Epoch  20  BCE = 0.4797
Epoch  40  BCE = 0.4047
Epoch  60  BCE = 0.3568
Epoch  80  BCE = 0.3257
Epoch 100  BCE = 0.3056
Epoch 120  BCE = 0.2939
Epoch 140  BCE = 0.2857
Epoch 160  BCE = 0.2787
Epoch 180  BCE = 0.2736
Epoch 200  BCE = 0.2686
Training complete.


### 12.2 Training with ELBOInference (variational)

In [37]:
pgm_asia_elbo = build_asia_pgm()
elbo_engine_asia = ELBOInference(pgm_asia_elbo, num_particles=2)
optimizer = torch.optim.Adam(elbo_engine_asia.parameters(), lr=1e-3)

pgm_asia_elbo.train()
for epoch in range(200):
    epoch_loss = 0.0
    n_batches  = 0
    for batch in dm.train_dataloader():
        x_b = batch['inputs']['x'].cpu()        # (B, 32)
        c_b = batch['concepts']['c'].float()    # (B, 8)

        targets = {name: c_b[:, CONCEPT_IDX[name], None] for name in concept_names}

        optimizer.zero_grad()
        result = elbo_engine_asia.query(
            query=concept_names,
            evidence={'input': x_b},
            targets=targets,
        )
        result.loss.backward()
        optimizer.step()
        epoch_loss += result.loss.item()
        n_batches  += 1

    if (epoch + 1) % 20 == 0:
        print(f'Epoch {epoch+1:3d}  neg-ELBO = {epoch_loss / n_batches:.4f}')

pgm_asia_elbo.eval()
print('Training complete.')


Epoch   5  neg-ELBO = 3322.8027
Epoch  10  neg-ELBO = 2964.7266
Epoch  15  neg-ELBO = 2741.1377
Epoch  20  neg-ELBO = 2565.0530
Epoch  25  neg-ELBO = 2414.3317
Epoch  30  neg-ELBO = 2282.4520
Epoch  35  neg-ELBO = 2165.7836
Epoch  40  neg-ELBO = 2060.6797
Epoch  45  neg-ELBO = 1966.9384
Epoch  50  neg-ELBO = 1884.2574
Epoch  55  neg-ELBO = 1805.6513
Epoch  60  neg-ELBO = 1738.2507
Epoch  65  neg-ELBO = 1674.9199
Epoch  70  neg-ELBO = 1617.2539
Epoch  75  neg-ELBO = 1564.1381
Epoch  80  neg-ELBO = 1517.5590
Epoch  85  neg-ELBO = 1477.6193
Epoch  90  neg-ELBO = 1437.1119
Epoch  95  neg-ELBO = 1400.7583
Epoch 100  neg-ELBO = 1366.1474
Epoch 105  neg-ELBO = 1337.4437
Epoch 110  neg-ELBO = 1309.0994
Epoch 115  neg-ELBO = 1282.2981
Epoch 120  neg-ELBO = 1258.7441
Epoch 125  neg-ELBO = 1237.4920
Epoch 130  neg-ELBO = 1216.0459
Epoch 135  neg-ELBO = 1195.4840
Epoch 140  neg-ELBO = 1179.5611
Epoch 145  neg-ELBO = 1163.9031
Epoch 150  neg-ELBO = 1148.7345
Epoch 155  neg-ELBO = 1136.3892
Epoch 16

### 12.3 Test-time queries

In [38]:
# Pull a real test mini-batch from the data module
test_batch = next(iter(dm.test_dataloader()))
X_test    = test_batch['inputs']['x'][:16].cpu()   # (16, 32)
x_single  = X_test[:1]                              # (1, 32) for posterior engines

# --- A. Forward query: predict all concept marginals from input embedding ---
with torch.no_grad():
    fwd = det_engine_asia.query(concept_names, evidence={'input': X_test}, return_probs=True)

# Model marginals: average predicted probability over the test batch
model_marginals = fwd.probs.mean(0)  # shape (8,)

print(f'Forward query — model vs empirical marginals (averaged over {len(X_test)} test samples)')
print(f"  {'concept':<8}  {'model':>8}  {'empirical':>10}  {'|Δ|':>6}")
print(f"  {'-'*40}")
for name, pred, emp in zip(concept_names, model_marginals.tolist(), emp_marginals.tolist()):
    print(f'  P({name:<6}) = {pred:8.3f}  {emp:10.3f}  {abs(pred - emp):6.3f}')


Forward query — model vs empirical marginals (averaged over 16 test samples)
  concept      model   empirical     |Δ|
  ----------------------------------------
  P(asia  ) =    0.993       0.991   0.002
  P(tub   ) =    0.978       0.991   0.013
  P(smoke ) =    0.485       0.506   0.021
  P(lung  ) =    0.939       0.945   0.006
  P(bronc ) =    0.542       0.552   0.010
  P(either) =    0.939       0.936   0.003
  P(xray  ) =    0.853       0.891   0.039
  P(dysp  ) =    0.573       0.567   0.005


In [39]:
# --- B. Backward query: P(smoke, bronc | input, dysp=1, xray=1) ---
imp = ImportanceQuery(pgm_asia)

back_evidence = {
    'input': x_single,
    'dysp':  torch.ones(1, 1),
    'xray':  torch.ones(1, 1),
}
back = imp.query(variables=['smoke', 'bronc'], evidence=back_evidence, num_samples=3000)

model_smoke_back = back.samples['smoke'].float().mean().item()
model_bronc_back = back.samples['bronc'].float().mean().item()

print('Backward IS query — model vs empirical (conditioned on dysp=1, xray=1)')
print(f"  {'concept':<8}  {'model':>8}  {'empirical':>10}  {'|Δ|':>6}")
print(f"  {'-'*40}")
print(f"  P({'smoke':<6}) = {model_smoke_back:8.3f}  {emp_smoke_given_dysp_xray.item():10.3f}  {abs(model_smoke_back - emp_smoke_given_dysp_xray.item()):6.3f}")
print(f"  P({'bronc':<6}) = {model_bronc_back:8.3f}  {emp_bronc_given_dysp_xray.item():10.3f}  {abs(model_bronc_back - emp_bronc_given_dysp_xray.item()):6.3f}")


Backward IS query — model vs empirical (conditioned on dysp=1, xray=1)
  concept      model   empirical     |Δ|
  ----------------------------------------
  P(smoke ) =    0.403       0.618   0.216
  P(bronc ) =    0.594       0.855   0.261


In [40]:
# --- C. Exact discrete posterior: P(either, lung, tub | input, xray=1) ---
exact = ExactDiscreteQuery(pgm_asia, temperature=1)

hidden_evidence = {'input': x_single, 'xray': torch.ones(1, 1)}
hidden = exact.query(variables=['either', 'lung', 'tub'], evidence=hidden_evidence, num_samples=200)

emp_hidden = {
    'either': emp_either_given_xray.item(),
    'lung':   emp_lung_given_xray.item(),
    'tub':    emp_tub_given_xray.item(),
}

print('Exact discrete posterior — model vs empirical (conditioned on xray=1)')
print(f"  {'concept':<8}  {'model':>8}  {'empirical':>10}  {'|Δ|':>6}")
print(f"  {'-'*40}")
for name in ['either', 'lung', 'tub']:
    if name in hidden.samples:
        model_p = hidden.samples[name].float().mean().item()
        emp_p   = emp_hidden[name]
        print(f"  P({name:<6}) = {model_p:8.3f}  {emp_p:10.3f}  {abs(model_p - emp_p):6.3f}")
    else:
        print(f"  P({name:<6}) = {'N/A':>8}")


Exact discrete posterior — model vs empirical (conditioned on xray=1)
  concept      model   empirical     |Δ|
  ----------------------------------------
  P(either) =    0.960       0.999   0.039
  P(lung  ) =    0.920       0.999   0.079
  P(tub   ) =    0.975       1.000   0.025


---
## 13. Key Design Tensions & Refactoring Notes

This section summarises the most important design choices, inconsistencies, and areas to address during a refactor. Items marked ✅ have been resolved; remaining items are open design tensions.

### 13.1 ✅ Two parallel forward loops — shared per-variable step

There are still two implementations of the topological forward pass:

| Location | Used by | Mechanism |
|---|---|---|
| `ForwardInference._predict_level()` | `DeterministicInference`, `AncestralSamplingInference`, `IndependentInference` | Custom PyTorch loop; supports parallelism, teacher forcing, interventions |
| `BayesianNetwork.forward()` | `ELBOInference`, `ImportanceQuery`, `ExactDiscreteQuery`, `MCMCQuery` | Pyro generative model; `pyro.plate`, `pyro.sample`, effect handlers |

Both loops keep the runtime features they need (CUDA/thread parallelism for the PyTorch loop; effect handlers for the Pyro loop), but they now share the per-variable propagation step via the new helper `_ProbabilisticModelBase._propagate_raw(var, raw_params, mode)`:

* `mode='pyro'` — registers `pyro.sample` / `pyro.deterministic` (used by `BayesianNetwork.forward`).
* `mode='deterministic'` — analytical mean / sigmoid / softmax (used by `DeterministicInference.activate`).
* `mode='ancestral'` — `rsample` / `sample` (used by `AncestralSamplingInference.activate`).

Combined with the already-shared `_run_cpd` and `_build_parent_kwargs`, this puts the *what-each-variable-does* logic in one place. Full unification of the two loops is intentionally deferred — the runtime features they offer are too different to merge cleanly.

### 13.2 ✅ `activate()` unified through `make_distribution()`

`Variable.activation` has been removed. `ForwardInference.activate()` now delegates to `variable.make_distribution(raw)` and uses the resulting distribution's analytical mean (with `sigmoid`/`softmax` fallbacks for relaxed distributions that lack a closed-form mean). The variable owns parameter→distribution mapping; the distribution owns the conversion to probabilities — a single source of truth.

### 13.3 ✅ `InferenceOutput` everywhere

All inference engines now return `InferenceOutput`:

* `ForwardInference.query()` (deterministic / ancestral / independent / ELBO) returns concatenated `(B, sum_sizes)` tensors in `.parameters` / `.probs` / `.joint` / `.loss`.
* `ImportanceQuery`, `ExactDiscreteQuery`, `MCMCQuery`, and `BayesianNetwork.query()` populate the new `.samples` field — a `Dict[str, Tensor(num_samples, *batch, size)]` — and set `.probs` to the empirical mean over the leading sample dimension (concatenated in the requested-variable order).

Use `result.samples['var']` for raw posterior draws and `result.probs` for a deterministic point estimate.

### 13.4 `ParametricCPD.__new__` magic

The `__new__` method on `ParametricCPD` (and `Variable`) intercepts `concepts=[...]` construction to return a `list` of independent CPDs (when `shared=False`) instead of a single instance. This is ergonomic but breaks the normal Python `isinstance` contract (`isinstance(ParametricCPD(concepts=[...], ...), ParametricCPD)` returns `False`). It also complicates subclassing.

The split between `concept=` (str) and `concepts=[...]` (list) makes the user-facing intent explicit, but the underlying list-construction trick remains.

### 13.5 String-parent references

CPD parents can be provided as strings and are resolved to `Variable` objects during `BayesianNetwork._initialize()` (after `_initialize_directed`, before topological sort). This makes the API ergonomic but adds a resolution step that can fail late (at model construction, not at CPD construction).

### 13.6 `shared` CPDs

A shared CPD (`shared=True` together with `concepts=[...]`) outputs concatenated parameters for multiple concepts simultaneously (e.g., a single `nn.Linear(D, N)` for N binary concepts). The output is sliced by `ForwardInference._predict_level()`. This is an important performance optimisation but adds bookkeeping complexity (`_shared_cpd_map`).

When `shared=True`, the CPD has both `.concept` (the canonical first name) and `.concepts` (the full list); when `shared=False`, only `.concept` is set.

### 13.7 ✅ Multi-parameter CPD outputs use a Dict container

For multi-parameter distributions (Normal: loc + scale; MultivariateNormal: loc + scale_tril), a `ModuleDict` parametrization now produces a `Dict[str, Tensor]` from `cpd.forward(...)` rather than a concatenated tensor. `Variable.make_distribution(...)` accepts both forms (Tensor → split by `_PARAM_DIMS`, Dict → keyed lookup).

### 13.8 ✅ `pyro_forward` renamed to `sample`

`ParametricCPD.pyro_forward(...)` is now `ParametricCPD.sample(...)`, aligning with Pyro's vocabulary (the method internally calls `pyro.sample`).


In [41]:
# Summary: full module tree of a BayesianNetwork
import torch

print('BayesianNetwork parameters (trainable):')
total = 0
for name, param in pgm_asia.named_parameters():
    print(f'  {name:50s} {tuple(param.shape)}')
    total += param.numel()
print(f'\nTotal: {total} parameters')

BayesianNetwork parameters (trainable):
  factors.asia.parametrization.weight                (1, 32)
  factors.asia.parametrization.bias                  (1,)
  factors.smoke.parametrization.weight               (1, 32)
  factors.smoke.parametrization.bias                 (1,)
  factors.tub.parametrization.weight                 (1, 1)
  factors.tub.parametrization.bias                   (1,)
  factors.lung.parametrization.weight                (1, 1)
  factors.lung.parametrization.bias                  (1,)
  factors.bronc.parametrization.weight               (1, 1)
  factors.bronc.parametrization.bias                 (1,)
  factors.either.parametrization.weight              (1, 2)
  factors.either.parametrization.bias                (1,)
  factors.xray.parametrization.weight                (1, 1)
  factors.xray.parametrization.bias                  (1,)
  factors.dysp.parametrization.weight                (1, 2)
  factors.dysp.parametrization.bias                  (1,)

Total: 80 par

In [42]:
# Quick import reference for contributors
print("""
Quick import reference
======================

from torch_concepts.nn.modules.mid.models.variable import (
    Variable, ConceptVariable, LatentVariable, ExogenousVariable,
    param_dim
)
from torch_concepts.nn.modules.mid.models.cpd import ParametricCPD
from torch_concepts.nn.modules.mid.models.probabilistic_model import (
    BayesianNetwork,
    ProbabilisticModel,       # alias for BayesianNetwork
    _ProbabilisticModelBase,  # abstract base
)
from torch_concepts.nn.modules.mid.inference.deterministic import DeterministicInference
from torch_concepts.nn.modules.mid.inference.ancestral import AncestralSamplingInference
from torch_concepts.nn.modules.mid.inference.independent import IndependentInference
from torch_concepts.nn.modules.mid.inference.elbo import ELBOInference
from torch_concepts.nn.modules.mid.inference.guide import AmortizedGuide
from torch_concepts.nn.modules.mid.inference.importance import ImportanceQuery
from torch_concepts.nn.modules.mid.inference.exact_discrete import ExactDiscreteQuery
from torch_concepts.nn.modules.mid.inference.mcmc import MCMCQuery
from torch_concepts.nn.modules.outputs import InferenceOutput
""")


Quick import reference

from torch_concepts.nn.modules.mid.models.variable import (
    Variable, ConceptVariable, LatentVariable, ExogenousVariable,
    param_dim
)
from torch_concepts.nn.modules.mid.models.cpd import ParametricCPD
from torch_concepts.nn.modules.mid.models.probabilistic_model import (
    BayesianNetwork,
    ProbabilisticModel,       # alias for BayesianNetwork
    _ProbabilisticModelBase,  # abstract base
)
from torch_concepts.nn.modules.mid.inference.deterministic import DeterministicInference
from torch_concepts.nn.modules.mid.inference.ancestral import AncestralSamplingInference
from torch_concepts.nn.modules.mid.inference.independent import IndependentInference
from torch_concepts.nn.modules.mid.inference.elbo import ELBOInference
from torch_concepts.nn.modules.mid.inference.guide import AmortizedGuide
from torch_concepts.nn.modules.mid.inference.importance import ImportanceQuery
from torch_concepts.nn.modules.mid.inference.exact_discrete import ExactDiscreteQu